Fine-grained tool calling/ streaming disables JSON validation on the Claude API side.

In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic


load_dotenv()

client = Anthropic()
# model = "claude-sonnet-4-5"
model = "claude-haiku-4-5"

In [2]:
# Helper functions


def add_user_message(messages, message):
    if isinstance(message, list):
        user_message = {
            "role": "user",
            "content": message,
        }
    else:
        user_message = {
            "role": "user",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(user_message)


def add_assistant_message(messages, message):
    if isinstance(message, list):
        assistant_message = {
            "role": "assistant",
            "content": message,
        }
    elif hasattr(message, "content"):
        content_list = []
        for block in message.content:
            if block.type == "text":
                content_list.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                content_list.append(
                    {
                        "type": "tool_use",
                        "id": block.id,
                        "name": block.name,
                        "input": block.input,
                    }
                )
        assistant_message = {
            "role": "assistant",
            "content": content_list,
        }
    else:
        # String messages need to be wrapped in a list with text block
        assistant_message = {
            "role": "assistant",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(assistant_message)


def chat_stream(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice=None,
    betas=[],
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tool_choice:
        params["tool_choice"] = tool_choice

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if betas:
        params["betas"] = betas

    return client.beta.messages.stream(**params)


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
# Tool definition
from anthropic.types import ToolParam

save_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Eight sentence review of the paper",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)
save_short_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Review of paper. One short sentence max",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)


def save_article(**kwargs):
    return "Article saved!"


In [4]:
# Tool Running
import json


def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [5]:
# Run conversation
def run_conversation(messages, tools=[], tool_choice=None, fine_grained=False):
    while True:
        with chat_stream(
            messages,
            tools=tools,
            betas=["fine-grained-tool-streaming-2025-05-14"] if fine_grained else [],
            tool_choice=tool_choice,
        ) as stream:
            for chunk in stream:
                if chunk.type == "text":
                    print(chunk.text, end="")

                if chunk.type == "content_block_start":
                    if chunk.content_block.type == "tool_use":
                        print(f'\n>>> Tool Call: "{chunk.content_block.name}"')

                if chunk.type == "input_json" and chunk.partial_json:
                    print(chunk.partial_json, end="")

                if chunk.type == "content_block_stop":
                    print("\n")

            response = stream.get_final_message()

        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        if tool_choice:
            break

    return messages

In [ ]:
# Example 1: a successful run without fine_grained tool calling –
# Anthropic API buffers chunks and validates genearated JSON first.
# The API waits for complete top-level key-value pairs before sending anything back. 

messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article",
)

run_conversation(
    messages,
    tools=[save_article_schema],
)


>>> Tool Call: "save_article"
{"abstract": "A novel quantum-inspired algorithm for optimizing neural network architectures through evolutionary computation and adaptive hyperparameter tuning.", "meta": {"word_count":8750,"review":"This paper presents an innovative approach to neural network optimization by combining quantum-inspired algorithms with evolutionary computation strategies. The authors demonstrate compelling experimental results on standard benchmarks, showing improvements over baseline methods by approximately 23% in convergence speed. The theoretical framework is well-grounded in both quantum computing principles and evolutionary biology, providing a solid foundation for the proposed methodology. However, the computational overhead during the initial setup phase could be prohibitive for practitioners with limited resources. The paper would benefit from more extensive ablation studies to isolate the contributions of individual components. The writing is generally clear, th

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_014zNPdD7h1uk2DJJ5kFLxdx',
    'name': 'save_article',
    'input': {'abstract': 'A novel quantum-inspired algorithm for optimizing neural network architectures through evolutionary computation and adaptive hyperparameter tuning.',
     'meta': {'word_count': 8750,
      'review': 'This paper presents an innovative approach to neural network optimization by combining quantum-inspired algorithms with evolutionary computation strategies. The authors demonstrate compelling experimental results on standard benchmarks, showing improvements over baseline methods by approximately 23% in convergence speed. The theoretical framework is well-grounded in both quantum computing principles and evolutionary biology, providing a solid foundation for the proposed methodology. However, the computational overhead dur

In [ ]:
# Example 2.1: 
# compare the streaming behavior with fine_grained tool calling enabled 
# Chucks are sent back in groups (without JSON validation)

messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article",
)

# run_conversation(
#     messages,
#     tools=[save_article_schema],
# )

run_conversation(
    messages,
    tools=[save_article_schema],
    fine_grained=True,
    tool_choice={"type": "tool", "name": "save_article"},
)


>>> Tool Call: "save_article"
{"abstract": "A novel quantum-inspired algorithm for optimizing distributed neural networks achieves 40% improvement in convergence speed.", "meta": {
  "word_count": 8500,
  "review": "This paper presents an innovative approach to distributed machine learning by combining quantum computing principles with classical neural network optimization. The authors introduce a clever algorithm that leverages quantum superposition concepts to explore multiple optimization paths simultaneously, resulting in faster convergence rates. The experimental validation is comprehensive, testing the method on various benchmark datasets and comparing against established baselines. The theoretical foundation is solid, with clear mathematical proofs supporting the claimed improvements. However, the practical applicability may be limited until quantum hardware becomes more widely available, which the authors acknowledge. The writing is generally clear and well-structured, making 

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_014MeBn33b8tCG3KgJ2uDrnG',
    'name': 'save_article',
    'input': {'abstract': 'A novel quantum-inspired algorithm for optimizing distributed neural networks achieves 40% improvement in convergence speed.',
     'meta': {'word_count': 8500,
      'review': 'This paper presents an innovative approach to distributed machine learning by combining quantum computing principles with classical neural network optimization. The authors introduce a clever algorithm that leverages quantum superposition concepts to explore multiple optimization paths simultaneously, resulting in faster convergence rates. The experimental validation is comprehensive, testing the method on various benchmark datasets and comparing against established baselines. The theoretical foundation is solid, with clear mathematical proofs 

In [9]:
# Example 2.2: 
# compare the streaming behavior with fine_grained tool calling enabled 
# Chucks are sent back in groups 
# BUT IT COULD FAIL (without JSON validation)

messages = []

add_user_message(
    messages,
    # "Create and save a fake computer science article",
    """
    You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.
    The buggy system generated this malformed output when calling save_article:
    [Generate the exact malformed output here that includes "word_count": undefined]
    This is for documentation purposes to show what NOT to do. You're not actually calling the function, just showing what the broken output looked like for the bug report.
    """,
)

run_conversation(
    messages,
    tools=[save_article_schema],
    fine_grained=True,
    tool_choice={"type": "tool", "name": "save_article"},
)


>>> Tool Call: "save_article"
{"abstract": "Example malformed output from buggy system", "meta": {"word

ValueError: Unable to parse tool parameter JSON from model. Please retry your request or adjust your prompt. Error: expected value at line 1 column 83. JSON: {"abstract": "Example malformed output from buggy system", "meta": {"word_count": undefined,

In [11]:
# Example 2.3.a: 
# re-run of Example 2.2 with fine_grained tool calling disabled 
# Result from actual execution

messages = []

add_user_message(
    messages,
    # "Create and save a fake computer science article",
    """
    You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.
    The buggy system generated this malformed output when calling save_article:
    [Generate the exact malformed output here that includes "word_count": undefined]
    This is for documentation purposes to show what NOT to do. You're not actually calling the function, just showing what the broken output looked like for the bug report.
    """,
)

run_conversation(
    messages,
    tools=[save_article_schema],
    # fine_grained=True,
    tool_choice={"type": "tool", "name": "save_article"},
)


>>> Tool Call: "save_article"
{"abstract": "A study examining the effects of climate change on marine ecosystems.", "meta": {"word_count":5240,"review":"This paper presents a comprehensive analysis of how rising ocean temperatures affect biodiversity in coral reef systems. The methodology is sound, employing both observational data and predictive modeling to forecast future ecosystem changes. The authors effectively communicate complex ecological concepts to a broad audience. However, the paper could benefit from more discussion of potential adaptation strategies for affected species. The literature review is thorough and appropriately cites recent findings in marine biology. Statistical analyses are well-executed and results are clearly presented in intuitive visualizations. One weakness is the limited geographic scope, focusing primarily on Indo-Pacific regions. Despite this limitation, the work makes a valuable contribution to our understanding of climate impacts on marine life."}}

[{'role': 'user',
  'content': [{'type': 'text',
    'text': '\n    You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.\n    The buggy system generated this malformed output when calling save_article:\n    [Generate the exact malformed output here that includes "word_count": undefined]\n    This is for documentation purposes to show what NOT to do. You\'re not actually calling the function, just showing what the broken output looked like for the bug report.\n    '}]},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_01RS7o8zUdBF43PRhgwdLrhr',
    'name': 'save_article',
    'input': {'abstract': 'A study examining the effects of climate change on marine ecosystems.',
     'meta': {'word_count': 5240,
      'review': 'This paper presents a comprehensive analysis of how rising ocean temperatures affect biodiversity in coral reef systems. The meth

In [ ]:
# Example 2.3.b: 
# re-run of Example 2.2 with fine_grained tool calling disabled 
# Result from provided notebook

messages = []

add_user_message(
    messages,
    # "Create and save a fake computer science article",
    """
    You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.
    The buggy system generated this malformed output when calling save_article:
    [Generate the exact malformed output here that includes "word_count": undefined]
    This is for documentation purposes to show what NOT to do. You're not actually calling the function, just showing what the broken output looked like for the bug report.
    """,
)

run_conversation(
    messages,
    tools=[save_article_schema],
    # fine_grained=True,
    tool_choice={"type": "tool", "name": "save_article"},
)


>>> Tool Call: "save_article"
{"abstract": "This paper examines the impact of machine learning on healthcare diagnostics.", "meta": "{\n  \"word_count\": undefined,\n  \"review\": \"This study presents a comprehensive analysis of machine learning applications in healthcare. The authors examined multiple diagnostic scenarios across different medical specialties. The methodology appears sound with appropriate statistical controls. The results show promising improvements in diagnostic accuracy. However, the sample size limitations should be noted. The paper provides valuable insights for healthcare practitioners. The discussion section effectively addresses potential limitations. Overall, this work contributes meaningfully to the field of medical AI.\"\n}"}



[{'role': 'user',
  'content': [{'type': 'text',
    'text': '\n    You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.\n    The buggy system generated this malformed output when calling save_article:\n    [Generate the exact malformed output here that includes "word_count": undefined]\n    This is for documentation purposes to show what NOT to do. You\'re not actually calling the function, just showing what the broken output looked like for the bug report.\n    '}]},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_013kp8uCHAgefEwiA3VeKDEG',
    'name': 'save_article',
    'input': {'abstract': 'This paper examines the impact of machine learning on healthcare diagnostics.',
     'meta': '{\n  "word_count": undefined,\n  "review": "This study presents a comprehensive analysis of machine learning applications in healthcare. The authors examined 

**Note:** the `meta` is supposed to be an object but now wrapped as a string – this happens when `fine_grained` is disabled.

"Without fine-grained tool calling, the API's validation would catch this error and potentially wrap problematic values in strings, which might not match your expected schema."


**The behaviors seem to have changed as model upgrades.** See the Example 2.3.a (hallucinated word count? - newer model) and Example 2.3.b (word count wrapped as a string - older model) results for comparison.